# Importing Libraries:

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader

# Importing The Dataset:

In [2]:
df= pd.read_csv('IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
data= df.copy()

# EDA

Understanding about the dataset and finding inconsistancis

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [6]:
df.duplicated().sum() # Checking for duplicate values.

418

In [7]:
df['review']

0        One of the other reviewers has mentioned that ...
1        A wonderful little production. <br /><br />The...
2        I thought this was a wonderful way to spend ti...
3        Basically there's a family where a little boy ...
4        Petter Mattei's "Love in the Time of Money" is...
                               ...                        
49995    I thought this movie did a down right good job...
49996    Bad plot, bad dialogue, bad acting, idiotic di...
49997    I am a Catholic taught in parochial elementary...
49998    I'm going to have to disagree with the previou...
49999    No one expects the Star Trek movies to be high...
Name: review, Length: 50000, dtype: object

 There are two columns, 1st is the review and based on the emotion the 2nd column is created.

    -- There are no null values.

    -- Duplicates are there, which need to be removed.

    -- Some inconsistencies are present like punctuations, stopwords, html tags etc.

# Data Preprocessing:

Removing all the inconsistancies

In [8]:
df.drop_duplicates(inplace=True) # Dropping duplicate values.

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49582 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     49582 non-null  object
 1   sentiment  49582 non-null  object
dtypes: object(2)
memory usage: 1.1+ MB


In [10]:
df['review']= df['review'].str.lower() # lower casing.

In [11]:
# Removing urls:
import re
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text

df["review"] = df["review"].apply(remove_urls)

In [12]:
# Remove html tags
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [13]:
# Removing Punctuations
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

In [15]:
# Tokenization & Removing StopWords
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [16]:
 # stemming
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [17]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti film techniqu unssum oldtim...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [18]:
# Encoding:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [19]:
df['sentiment'] # Positive->1, negative->0.

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int32

In [20]:
# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [21]:
# Dataset And DataLoader Creation:
from sklearn.model_selection import train_test_split

y= df['sentiment']
# Splitting:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# converting sparse matrix to array:
X_train = X_train.toarray()
X_test = X_test.toarray()


# Train & Test set:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

# Train & Test Loader:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

# Building Model:

In [22]:
import torch.optim as optim
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out


input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Training of RNN:

In [23]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.1959259808063507
epoch = 2/10 and loss = 0.27920839190483093
epoch = 3/10 and loss = 0.17403730750083923
epoch = 4/10 and loss = 0.300565630197525
epoch = 5/10 and loss = 0.3928448557853699
epoch = 6/10 and loss = 0.1835182160139084
epoch = 7/10 and loss = 0.2976030111312866
epoch = 8/10 and loss = 0.19567236304283142
epoch = 9/10 and loss = 0.33638569712638855
epoch = 10/10 and loss = 0.2895983159542084


In [24]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.57023293334677


# Deployment:

Saving Models

In [25]:
import pickle
import torch

# Save vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tf, f)

# Save model
torch.save(model.state_dict(), "sentiment_model.pth")

Load for interface

In [26]:
# Load vectorizer
with open("tfidf_vectorizer.pkl", "rb") as f:
    tf = pickle.load(f)

# Load model
model = RNN(input_size=5000)
model.load_state_dict(torch.load("sentiment_model.pth"))
model.eval()

RNN(
  (rnn): RNN(5000, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

Pre-Processing Function

In [27]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)

    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))

    tokens = [word for word in tokens if word not in stop_words]

    ps = PorterStemmer()
    tokens = [ps.stem(word) for word in tokens]

    return " ".join(tokens)

Predicted Function

In [28]:
def predict_sentiment(text):
    text = preprocess_text(text)

    vector = tf.transform([text]).toarray()
    tensor = torch.from_numpy(vector).float()

    with torch.no_grad():
        tensor = tensor.unsqueeze(1)
        output = model(tensor)
        prob = torch.sigmoid(output).item()

    if prob > 0.5:
        return f"Positive 😊 (Confidence: {prob:.2f})"
    else:
        return f"Negative 😞 (Confidence: {1-prob:.2f})"

UI

In [29]:
import gradio as gr

def analyze_sentiment(text):
    return predict_sentiment(text)

interface = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Enter your sentence here..."
    ),
    outputs="text",
    title="🎬 IMDB Sentiment Analyzer",
    description="Enter a movie review and get sentiment prediction using RNN + TF-IDF"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
